<a href="https://colab.research.google.com/github/vs-06/git-test/blob/main/MSDS_Project_Team3_CLIP_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pillow torch transformers



## Loading Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/flickr30k_data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Dataset folder ready:", DATA_DIR)


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "AnyModal/flickr30k",
    split="train",
    cache_dir=DATA_DIR
)

print("Dataset loaded and cached.")




In [ ]:
ds.save_to_disk(DATA_DIR + "/arrow_dataset")


## FUTURE DATA LOADS:


In [ ]:
from datasets import load_from_disk
ds = load_from_disk(DATA_DIR + "/arrow_dataset")

In [ ]:
ex = ds[0]
for k, v in ex.items():
    if isinstance(v, str):
        print("string field:", k)
    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], str):
        print("list-of-strings field:", k, "len:", len(v))


## Step 6: Choosing the text field

In [ ]:
CAPTION_KEY = "original_alt_text"

ex = ds[0]
print("Example captions:")
for i, c in enumerate(ex[CAPTION_KEY]):
    print(i, c)


## Creating Labelled Pairs (Positive + Negative)

In [ ]:
import random

CAPTION_KEY = "original_alt_text"
random.seed(42)

def make_pair(idx):
    ex = ds[idx]
    image = ex["image"]
    pos_text = random.choice(ex[CAPTION_KEY])

    # Pick a different image for the negative caption
    neg_idx = random.randrange(len(ds))
    while neg_idx == idx:
        neg_idx = random.randrange(len(ds))

    neg_text = random.choice(ds[neg_idx][CAPTION_KEY])

    return (image, pos_text, 1), (image, neg_text, 0)

pos, neg = make_pair(0)
print("POS:", pos[1], "label:", pos[2])
print("NEG:", neg[1], "label:", neg[2])



## Step 8: Build a small Labelled Dataset pairs (image, text, label) pairs we can feed in to the CLIP later.

In [ ]:
import random

CAPTION_KEY = "original_alt_text"
random.seed(42)

def make_pair(idx):
    ex = ds[idx]
    image = ex["image"]
    pos_text = random.choice(ex[CAPTION_KEY])

    # negative caption from a different image
    neg_idx = random.randrange(len(ds))
    while neg_idx == idx:
        neg_idx = random.randrange(len(ds))
    neg_text = random.choice(ds[neg_idx][CAPTION_KEY])

    return (image, pos_text, 1), (image, neg_text, 0)

# Build N images -> 2N pairs (1 pos + 1 neg each)
N_IMAGES = 2000  # keep small for now
pairs = []
for idx in range(N_IMAGES):
    pos, neg = make_pair(idx)
    pairs.append(pos)
    pairs.append(neg)

random.shuffle(pairs)

print("Total pairs:", len(pairs))
print("Example:", pairs[0][1], "label:", pairs[0][2])


# Displaying Sample images with positive and negative captions.

In [ ]:
import matplotlib.pyplot as plt

def compare_pos_neg(pos, neg):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(pos[0])
    axes[0].set_title("Positive (1)")
    axes[0].axis("off")

    axes[1].imshow(neg[0])
    axes[1].set_title("Negative (0)")
    axes[1].axis("off")

    plt.show()

    print("Positive caption:", pos[1])
    print("Negative caption:", neg[1])

# Example
pos, neg = make_pair(10)
compare_pos_neg(pos, neg)


In [ ]:
import matplotlib.pyplot as plt

def inspect_raw_pairs(pairs, n=5):
    for i in range(n):
        image, text, label = pairs[i]

        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"Pair {i} | Label: {label}")
        plt.show()

        print("Caption:")
        print(text)
        print("Image type:", type(image))
        print("Image size:", image.size)
        print("=" * 80)

inspect_raw_pairs(pairs)


## Running ONE pair through CLIP and computing Euclidian Distance

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def get_embeddings(image, text):
    inputs = clip_processor(
        images=image,
        text=[text],
        return_tensors="pt",
        padding=True
    ).to(DEVICE)

    outputs = clip_model(**inputs)  # forward pass through both encoders

    # These are tensors of shape (batch, embed_dim)
    v_img = outputs.image_embeds[0]  # shape (512,)
    v_txt = outputs.text_embeds[0]   # shape (512,)

    return v_img, v_txt

def l2_distance(v1, v2):
    return torch.norm(v1 - v2, p=2).item()

pos_sample = next(s for s in pairs if s[2] == 1)
neg_sample = next(s for s in pairs if s[2] == 0)

vimg_pos, vtxt_pos = get_embeddings(pos_sample[0], pos_sample[1])
vimg_neg, vtxt_neg = get_embeddings(neg_sample[0], neg_sample[1])

print("Positive label=1 L2 distance:", l2_distance(vimg_pos, vtxt_pos))
print("Negative label=0 L2 distance:", l2_distance(vimg_neg, vtxt_neg))
print("Embedding dim:", vimg_pos.shape)


## Step 10: Computing Distances for 200 Positives and 200 Negatives

In [ ]:
import random
import numpy as np
import torch

def sample_by_label(pairs, label, k):
    subset = [p for p in pairs if p[2] == label]
    return random.sample(subset, k)

@torch.no_grad()
def compute_l2_for_samples(samples):
    dists = []
    for image, text, label in samples:
        v_img, v_txt = get_embeddings(image, text)
        dists.append(l2_distance(v_img, v_txt))
    return np.array(dists)

random.seed(42)

K = 200
pos_samples = sample_by_label(pairs, 1, K)
neg_samples = sample_by_label(pairs, 0, K)

pos_d = compute_l2_for_samples(pos_samples)
neg_d = compute_l2_for_samples(neg_samples)

print("POS mean/std:", pos_d.mean(), pos_d.std())
print("NEG mean/std:", neg_d.mean(), neg_d.std())
print("POS median:", np.median(pos_d))
print("NEG median:", np.median(neg_d))
print("Fraction where POS < NEG (pairwise by index):", np.mean(pos_d < neg_d))


## Step 11: Deterministic “one caption per image” setup (full dataset)

In [ ]:
import random, numpy as np, torch

CAPTION_KEY = "original_alt_text"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

N = len(ds)
print("Dataset size:", N)

# Choose ONE caption per image deterministically (reproducible)
def choose_caption(ex, rng):
    caps = ex[CAPTION_KEY]  # list[str]
    return caps[rng.randrange(len(caps))]

rng = random.Random(SEED)
chosen_captions = [choose_caption(ds[i], rng) for i in range(N)]

print("Example chosen captions:")
for k in range(3):
    print(f"[{k}] {chosen_captions[k]}")

## Step 12: Compute and Cache CLIP Embeddings

In [ ]:
from numpy.lib.format import open_memmap

In [ ]:
import os
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel

# Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Directory to store embeddings
EMB_DIR = os.path.join(DATA_DIR, "clip_embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

img_emb_path = os.path.join(EMB_DIR, "image_embeds.npy")
txt_emb_path = os.path.join(EMB_DIR, "text_embeds.npy")

BATCH_SIZE = 64  # Reduce to 32 if GPU memory error
EMB_DIM = 512

def l2_normalize(x, eps=1e-12):
    return x / (x.norm(p=2, dim=-1, keepdim=True) + eps)

@torch.no_grad()
def embed_batch(images, texts):
    inputs = clip_processor(
    images=images,
    text=texts,
    return_tensors="pt",
    padding=True,
    truncation=True,      # Truncation to avoid the break due to text token length limit
    max_length=77         # enforce CLIP limit
).to(DEVICE)

    outputs = clip_model(**inputs)

    img_vec = l2_normalize(outputs.image_embeds)
    txt_vec = l2_normalize(outputs.text_embeds)

    return img_vec.cpu().numpy(), txt_vec.cpu().numpy()


# If embeddings already computed, load them
if os.path.exists(img_emb_path) and os.path.exists(txt_emb_path):
    print("Loading cached embeddings...")
    image_embeds = np.load(img_emb_path, mmap_mode="r")
    text_embeds  = np.load(txt_emb_path, mmap_mode="r")
else:
    print("Computing embeddings for full dataset...")

    N = len(ds)
    image_embeds = open_memmap(img_emb_path, mode="w+", dtype="float32", shape=(N, EMB_DIM))
    text_embeds  = open_memmap(txt_emb_path, mode="w+", dtype="float32", shape=(N, EMB_DIM))
    for start in tqdm(range(0, N, BATCH_SIZE)):
        end = min(start + BATCH_SIZE, N)

        batch_images = [ds[i]["image"] for i in range(start, end)]
        batch_texts  = [chosen_captions[i] for i in range(start, end)]

        img_vecs, txt_vecs = embed_batch(batch_images, batch_texts)

        image_embeds[start:end] = img_vecs.astype("float32")
        text_embeds[start:end]  = txt_vecs.astype("float32")

    image_embeds.flush()
    text_embeds.flush()

    image_embeds = np.load(img_emb_path, mmap_mode="r")
    text_embeds  = np.load(txt_emb_path, mmap_mode="r")

print("Done.")
print("Image embeddings shape:", image_embeds.shape)
print("Text embeddings shape:", text_embeds.shape)

In [ ]:
import os

if os.path.exists(img_emb_path):
    os.remove(img_emb_path)

if os.path.exists(txt_emb_path):
    os.remove(txt_emb_path)

print("Old embedding files removed.")

## Cell to load cached embeddings.

In [ ]:
import os
import numpy as np

EMB_DIR = os.path.join(DATA_DIR, "clip_embeddings")

img_emb_path = os.path.join(EMB_DIR, "image_embeds.npy")
txt_emb_path = os.path.join(EMB_DIR, "text_embeds.npy")

# Memory-mapped load (fast + low RAM usage)
image_embeds = np.load(img_emb_path, mmap_mode="r")
text_embeds  = np.load(txt_emb_path, mmap_mode="r")

print("Loaded embeddings:")
print("Image shape:", image_embeds.shape)
print("Text shape:", text_embeds.shape)

## Step 13: Peat at the vectors (Sanity Checks + Quick Validity)

In [ ]:
import numpy as np

def summarize(name, X, n=5000):
    n = min(len(X), n)
    sample = np.array(X[:n], dtype=np.float32)

    norms = np.linalg.norm(sample, axis=1)
    print(f"\n{name}")
    print(" shape:", X.shape)
    print(" any NaNs?:", np.isnan(sample).any())
    print(" norms -> mean/std/min/max:",
          float(norms.mean()), float(norms.std()), float(norms.min()), float(norms.max()))
    print(" first vector (first 10 dims):", sample[0][:10])

summarize("image_embeds", image_embeds)
summarize("text_embeds", text_embeds)

# Quick similarity sanity: dot product of matching pairs (since we normalized)
dots = (np.array(image_embeds[:2000]) * np.array(text_embeds[:2000])).sum(axis=1)
print("\nMatching pair cosine (dot) over first 2000:")
print(" mean/std/min/max:", float(dots.mean()), float(dots.std()), float(dots.min()), float(dots.max()))

## Step 14: Create label 0 pairs + compare distance/cosine distributions

In [ ]:
import numpy as np

K = 5000  # you can set 2000 first if you want faster
SEED = 42
rng = np.random.default_rng(SEED)

N = len(image_embeds)

# sample K indices for positives
pos_i = rng.choice(N, size=K, replace=False)
pos_j = pos_i  # matching caption

# sample K indices for negatives: j != i
neg_i = pos_i.copy()
neg_j = rng.choice(N, size=K, replace=True)
mask = (neg_j == neg_i)
while mask.any():
    neg_j[mask] = rng.choice(N, size=mask.sum(), replace=True)
    mask = (neg_j == neg_i)

# gather vectors
Vi_pos = np.array(image_embeds[pos_i], dtype=np.float32)
Vt_pos = np.array(text_embeds[pos_j], dtype=np.float32)

Vi_neg = np.array(image_embeds[neg_i], dtype=np.float32)
Vt_neg = np.array(text_embeds[neg_j], dtype=np.float32)

# cosine (dot product since normalized)
cos_pos = (Vi_pos * Vt_pos).sum(axis=1)
cos_neg = (Vi_neg * Vt_neg).sum(axis=1)

# Euclidean distance
l2_pos = np.linalg.norm(Vi_pos - Vt_pos, axis=1)
l2_neg = np.linalg.norm(Vi_neg - Vt_neg, axis=1)

print("Cosine similarity:")
print("  pos mean/std:", float(cos_pos.mean()), float(cos_pos.std()))
print("  neg mean/std:", float(cos_neg.mean()), float(cos_neg.std()))
print("  P(pos > neg):", float((cos_pos > cos_neg).mean()))

print("\nEuclidean distance:")
print("  pos mean/std:", float(l2_pos.mean()), float(l2_pos.std()))
print("  neg mean/std:", float(l2_neg.mean()), float(l2_neg.std()))
print("  P(pos < neg):", float((l2_pos < l2_neg).mean()))

## ROC-AUC + PR-AUC

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

# -----------------------------
# Build positive and negative scores
# -----------------------------
K = 5000
SEED = 42
rng = np.random.default_rng(SEED)

N = len(image_embeds)

# Positive pairs: (i, i)
pos_i = rng.choice(N, size=K, replace=False)
pos_j = pos_i

# Negative pairs: (i, j) where j != i
neg_i = pos_i.copy()
neg_j = rng.choice(N, size=K, replace=True)
mask = (neg_j == neg_i)
while mask.any():
    neg_j[mask] = rng.choice(N, size=mask.sum(), replace=True)
    mask = (neg_j == neg_i)

Vi_pos = np.array(image_embeds[pos_i], dtype=np.float32)
Vt_pos = np.array(text_embeds[pos_j], dtype=np.float32)

Vi_neg = np.array(image_embeds[neg_i], dtype=np.float32)
Vt_neg = np.array(text_embeds[neg_j], dtype=np.float32)

# Cosine similarity because embeddings are normalized
cos_pos = (Vi_pos * Vt_pos).sum(axis=1)
cos_neg = (Vi_neg * Vt_neg).sum(axis=1)

# Labels and scores
y_true = np.concatenate([np.ones(K), np.zeros(K)])
scores = np.concatenate([cos_pos, cos_neg])  # higher = more likely positive

# -----------------------------
# Metrics
# -----------------------------
roc_auc = roc_auc_score(y_true, scores)
pr_auc = average_precision_score(y_true, scores)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

# -----------------------------
# ROC Curve
# -----------------------------
fpr, tpr, _ = roc_curve(y_true, scores)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - CLIP Cosine Baseline")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# -----------------------------
# PR Curve
# -----------------------------
precision, recall, _ = precision_recall_curve(y_true, scores)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - CLIP Cosine Baseline")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# -----------------------------
# Find best threshold by F1
# -----------------------------
thresholds = np.linspace(scores.min(), scores.max(), 300)

best_f1 = -1
best_thr = None

for thr in thresholds:
    y_pred_tmp = (scores >= thr).astype(int)
    f1 = f1_score(y_true, y_pred_tmp)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

# Final predictions
y_pred = (scores >= best_thr).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Best threshold: {best_thr:.4f}")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print("Confusion Matrix:")
print(cm)

# -----------------------------
# Plot confusion matrix
# -----------------------------
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Mismatch (0)", "Match (1)"])
disp.plot()
plt.title(f"Confusion Matrix (threshold = {best_thr:.4f})")
plt.show()

## Recall@K

In [ ]:
import numpy as np

def recall_at_k(image_embeds, text_embeds, ks=(1, 5, 10), subset=1000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(image_embeds), size=subset, replace=False)

    img = np.array(image_embeds[idx], dtype=np.float32)
    txt = np.array(text_embeds[idx], dtype=np.float32)

    # Similarity matrix: since normalized, dot product = cosine similarity
    S = img @ txt.T  # shape: (subset, subset)

    recalls = {}
    for k in ks:
        correct = 0
        for i in range(subset):
            topk = np.argsort(-S[i])[:k]
            if i in topk:
                correct += 1
        recalls[k] = correct / subset

    # Median rank of true caption
    ranks = []
    for i in range(subset):
        ranking = np.argsort(-S[i])
        rank_i = np.where(ranking == i)[0][0] + 1  # 1-based rank
        ranks.append(rank_i)

    median_rank = np.median(ranks)
    mean_rank = np.mean(ranks)

    return recalls, median_rank, mean_rank

recalls, median_rank, mean_rank = recall_at_k(
    image_embeds=image_embeds,
    text_embeds=text_embeds,
    ks=(1, 5, 10),
    subset=1000,
    seed=42
)

print("Recall@K:")
for k, v in recalls.items():
    print(f"Recall@{k}: {v:.4f}")

print(f"Median Rank: {median_rank}")
print(f"Mean Rank  : {mean_rank:.2f}")

## Embedding Scatter/ PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

subset = 300
SEED = 42
rng = np.random.default_rng(SEED)

idx = rng.choice(len(image_embeds), size=subset, replace=False)

img = np.array(image_embeds[idx], dtype=np.float32)
txt = np.array(text_embeds[idx], dtype=np.float32)

# Stack and reduce to 2D
X = np.vstack([img, txt])   # shape: (2*subset, 512)
pca = PCA(n_components=2)
X2 = pca.fit_transform(X)

img_2d = X2[:subset]
txt_2d = X2[subset:]

plt.figure(figsize=(8, 6))
plt.scatter(img_2d[:, 0], img_2d[:, 1], alpha=0.7, label="Image")
plt.scatter(txt_2d[:, 0], txt_2d[:, 1], alpha=0.7, label="Text")

# Optional: connect first 20 matched pairs
for i in range(min(20, subset)):
    plt.plot(
        [img_2d[i, 0], txt_2d[i, 0]],
        [img_2d[i, 1], txt_2d[i, 1]],
        alpha=0.3
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection of CLIP Image and Text Embeddings")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())